In [3]:
import pandas as pd
import numpy as np
from collections import defaultdict
from copy import deepcopy
from math import ceil

NUM_ORDERS = 6

itemtypes  = pd.read_csv("order_itemtypes.csv",  header=None, nrows=NUM_ORDERS)
quantities = pd.read_csv("order_quantities.csv", header=None, nrows=NUM_ORDERS)
totes_csv  = pd.read_csv("orders_totes.csv",     header=None, nrows=NUM_ORDERS)

# Build per-order demand, per-tote inventory, and unit records
order_demand   = defaultdict(lambda: defaultdict(int))
tote_inventory = defaultdict(lambda: defaultdict(int))
units = []

for o in range(itemtypes.shape[0]):
    for k in range(itemtypes.shape[1]):
        it = itemtypes.iat[o, k]
        qt = quantities.iat[o, k] if k < quantities.shape[1] else pd.NA
        tt = totes_csv.iat[o, k]  if k < totes_csv.shape[1]  else pd.NA
        if pd.notna(it) and pd.notna(qt) and pd.notna(tt):
            it, qt, tt = int(it), int(qt), int(tt)
            order_demand[o + 1][it]  += qt
            tote_inventory[tt][it]   += qt
            for _ in range(qt):
                units.append({"order": o + 1, "item_type": it, "tote": tt})

units_df  = pd.DataFrame(units)
orders    = sorted(order_demand.keys())
tote_ids  = sorted(tote_inventory.keys())

print("Order demand (item_type: qty):")
for o in orders:
    parts = [f"item {it} x {q}" for it, q in sorted(order_demand[o].items())]
    print(f"  Order {o}: {', '.join(parts)}")

print("\nTote inventory:")
for t in tote_ids:
    parts = [f"item {it} x {q}" for it, q in sorted(tote_inventory[t].items())]
    print(f"  Tote {t}: {', '.join(parts)}")

print(f"\nTotal units : {len(units_df)}")
print(f"Total orders: {len(orders)}")
print(f"Total totes : {len(tote_ids)}")

Order demand (item_type: qty):
  Order 1: item 1 x 2, item 3 x 3
  Order 2: item 2 x 1, item 3 x 3, item 4 x 1
  Order 3: item 5 x 2
  Order 4: item 0 x 3, item 5 x 1
  Order 5: item 1 x 1, item 2 x 1
  Order 6: item 1 x 2

Tote inventory:
  Tote 0: item 1 x 2, item 3 x 3
  Tote 1: item 1 x 2
  Tote 4: item 3 x 3
  Tote 7: item 5 x 2
  Tote 9: item 0 x 3
  Tote 10: item 5 x 1
  Tote 11: item 1 x 1
  Tote 12: item 2 x 1
  Tote 14: item 2 x 1, item 4 x 1

Total units : 20
Total orders: 6
Total totes : 9


In [4]:
NUM_CONVEYORS = 4

# --- FIFO: totes released in ascending tote-ID order (no reordering) ---
tote_sequence = sorted(tote_ids)

print("FIFO tote release sequence (ascending tote ID):")
print(f"  {tote_sequence}")

# --- Conveyor assignment: round-robin in ascending order-ID order ---
# FIFO does no optimisation; orders are assigned in the order they arrive (1,2,...).
order_sequence = sorted(orders)
order_to_conveyor = {}
conveyor_queues   = defaultdict(list)

for i, o in enumerate(order_sequence):
    c = (i % NUM_CONVEYORS) + 1
    order_to_conveyor[o] = c
    conveyor_queues[c].append(o)

print("\nConveyor assignment (round-robin, FIFO order):")
for c in sorted(conveyor_queues):
    queue_str = " -> ".join(str(o) for o in conveyor_queues[c])
    print(f"  Conveyor {c}: [{queue_str}]")

FIFO tote release sequence (ascending tote ID):
  [0, 1, 4, 7, 9, 10, 11, 12, 14]

Conveyor assignment (round-robin, FIFO order):
  Conveyor 1: [1 -> 5]
  Conveyor 2: [2 -> 6]
  Conveyor 3: [3]
  Conveyor 4: [4]


In [5]:
SLOT_TIME = 1   # seconds per release slot
BELT_TIME = 2   # seconds per conveyor traversal

# Build release-slot assignments from FIFO tote sequence.
# Units from each tote are released in contiguous slots, in the order
# they appear in units_df (preserving original CSV row order within a tote).
slot_assignments = []
slot = 0

for t in tote_sequence:
    tote_units = units_df[units_df["tote"] == t].copy()
    for _, u in tote_units.iterrows():
        slot_assignments.append({
            "slot":      slot,
            "order":     u["order"],
            "item_type": u["item_type"],
            "tote":      u["tote"],
            "conveyor":  order_to_conveyor[u["order"]],
        })
        slot += 1

schedule = pd.DataFrame(slot_assignments)

# Compute recirculation-aware pick times.
# pick_time = slot*SLOT_TIME + BELT_TIME + (c-1)*BELT_TIME + BELT_TIME/2 + 4*BELT_TIME*k
# k is the minimum non-negative integer such that pick_time >= conveyor_ready[c]
conveyor_ready = {c: 0.0 for c in range(1, NUM_CONVEYORS + 1)}

pick_times   = []
recirc_loops = []

for _, row in schedule.iterrows():
    s = row["slot"]
    c = row["conveyor"]

    first_arrival = s * SLOT_TIME + BELT_TIME + (c - 1) * BELT_TIME + BELT_TIME / 2.0
    loop_period   = 4 * BELT_TIME

    ready = conveyor_ready[c]
    if first_arrival >= ready:
        pick_time = first_arrival
        k = 0
    else:
        k = ceil((ready - first_arrival) / loop_period)
        pick_time = first_arrival + k * loop_period

    pick_times.append(pick_time)
    recirc_loops.append(k)
    conveyor_ready[c] = pick_time

schedule["pick_time"]    = pick_times
schedule["recirc_loops"] = recirc_loops
schedule["release_time"] = schedule["slot"] * SLOT_TIME

# Order-level timing
order_completion = schedule.groupby("order")["pick_time"].max()
order_start      = schedule.groupby("order")["pick_time"].min()
objective        = order_completion.sum()

print("Release schedule with pick times:")
print(schedule[["slot","release_time","pick_time","order","item_type","tote","conveyor","recirc_loops"]].to_string(index=False))

print("\nOrder timing:")
for o in order_sequence:
    c = order_to_conveyor[o]
    print(f"  Order {o} (conv {c}): start={order_start[o]:.1f}s, completion={order_completion[o]:.1f}s")

print(f"\nObjective (sum of completion times): {objective:.1f}s")

Release schedule with pick times:
 slot  release_time  pick_time  order  item_type  tote  conveyor  recirc_loops
    0             0        3.0      1          3     0         1             0
    1             1        4.0      1          3     0         1             0
    2             2        5.0      1          3     0         1             0
    3             3        6.0      1          1     0         1             0
    4             4        7.0      1          1     0         1             0
    5             5       10.0      6          1     1         2             0
    6             6       11.0      6          1     1         2             0
    7             7       12.0      2          3     4         2             0
    8             8       13.0      2          3     4         2             0
    9             9       14.0      2          3     4         2             0
   10            10       17.0      3          5     7         3             0
   11            1

In [6]:
ITEM_COLS = ["circle", "pentagon", "trapezoid", "triangle", "star", "moon", "heart", "cross"]

# Sort output rows by order start time (first pick), matching the other heuristics
order_start_times = schedule.groupby("order")["pick_time"].min().to_dict()
output_order = sorted(order_sequence, key=lambda o: order_start_times.get(o, float("inf")))

csv_rows = []
for o in output_order:
    conv = order_to_conveyor[o]
    row  = {"conv_num": conv}
    for it in range(8):
        row[it] = order_demand[o].get(it, 0)
    csv_rows.append(row)

out = pd.DataFrame(csv_rows)
out = out[["conv_num"] + list(range(8))]
out.columns = ["conv_num"] + ITEM_COLS

output_file = "fifo_baseline_output.csv"
out.to_csv(output_file, index=False)

print(f"Generated: {output_file}")
print(out.to_string(index=False))

print(f"\nOrder -> Conveyor (FIFO):")
for o in output_order:
    print(f"  Order {o} -> Conveyor {order_to_conveyor[o]}")

print(f"\nFIFO objective (sum of completion times): {objective:.1f}s")

Generated: fifo_baseline_output.csv
 conv_num  circle  pentagon  trapezoid  triangle  star  moon  heart  cross
        1       0         2          0         3     0     0      0      0
        2       0         2          0         0     0     0      0      0
        2       0         0          1         3     1     0      0      0
        3       0         0          0         0     0     2      0      0
        1       0         1          1         0     0     0      0      0
        4       3         0          0         0     0     1      0      0

Order -> Conveyor (FIFO):
  Order 1 -> Conveyor 1
  Order 6 -> Conveyor 2
  Order 2 -> Conveyor 2
  Order 3 -> Conveyor 3
  Order 5 -> Conveyor 1
  Order 4 -> Conveyor 4

FIFO objective (sum of completion times): 104.0s
